# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Adtividad en Equipos Semanas 7 y 8 : LDA y LMM audio-a-texto**

* **Nombres y matrículas:**

- Gabriela del Carmen González Domínguez - A01796282
- Bertha Itzel Salamanca Murcia - A01797439
- Omar Aguilar Macedo - A01797078


* **Número de Equipo:**
20

* ##### **En cada ejercicio pueden importar los paquetes o librerías que requieran.**

* ##### **En cada ejercicio pueden incluir las celdas y líneas de código que deseen.**

# **Ejercicio 1:**

* #### **Liga de los audios de las fábulas de Esopo:** https://www.gutenberg.org/ebooks/21144

* #### **Descargar los 10 archivos de audio solicitados: 1, 4, 5, 6, 14, 22, 24, 25, 26, 27.**



In [63]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.5 MB/s eta 0:00:00


In [37]:
#@title Imports
import requests
from pathlib import Path

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from transformers.utils import logging

import librosa
import torch

import re
import nltk
from nltk.corpus import stopwords

In [2]:
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')

os.environ["HF_TOKEN"] = hf_token

print("Hugging Face token loaded")

Hugging Face token loaded


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [4]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

def download_gutenberg_mp3(book_id: int, sub_file_id: int, extension: str = 'mp3', output_dir: str = "."):
    # Target URL pattern for Project Gutenberg MP3 files
    file_name = f"{book_id}-{sub_file_id:02d}.{extension}"
    base_url = f"https://gutenberg.org/files/{book_id}/{extension}/{file_name}"

    local_filename = Path(output_dir) / file_name

    if local_filename.exists():
        print(f"File {file_name} already exists. Skipping download.")
        return

    print(f"Downloading MP3 audio: {file_name}")
    try:
        response = requests.get(base_url, stream=True)
        response.raise_for_status()

        with open(local_filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"saved audio to: {local_filename}")

    except requests.exceptions.HTTPError as e:
        print(f"Failed to download. Book {book_id} might not have an MP3 version available. Error: {e}")


In [5]:
lista_fabulas_audio = [1, 4, 5, 6, 14, 22, 24, 25, 26, 27]
extension = 'mp3'
prefix = '21144'

for fabula in lista_fabulas_audio:
    download_gutenberg_mp3(prefix, fabula, extension)

File 21144-01.mp3 already exists. Skipping download.
File 21144-04.mp3 already exists. Skipping download.
File 21144-05.mp3 already exists. Skipping download.
File 21144-06.mp3 already exists. Skipping download.
File 21144-14.mp3 already exists. Skipping download.
File 21144-22.mp3 already exists. Skipping download.
File 21144-24.mp3 already exists. Skipping download.
File 21144-25.mp3 already exists. Skipping download.
File 21144-26.mp3 already exists. Skipping download.
File 21144-27.mp3 already exists. Skipping download.


# **Ejercicio 2a:**

* #### **Comenten el por qué del modelo seleccionado para extracción del texto de los audios.**

* #### **Extraer el contenido de los audios en texto.**

* #### **Sugerencia:** pueden extraerlo en un formato de diccionario, clave:valor $→$ {audio01:fabula01, ...}

### Ejemplo Wav2Vec

In [6]:
# Quitamos la barra de progreso cuando se descarga un modelo
logging.disable_progress_bar()

In [7]:
def transcribe_sample_wav2vec(audio, model_name):
    processor = Wav2Vec2Processor.from_pretrained(model_name)
    model = Wav2Vec2ForCTC.from_pretrained(model_name).to(device)
    input_values = processor(
        audio,
        sampling_rate=sample_rate,
        return_tensors="pt"
    ).input_values.to(device)
    with torch.no_grad():
        logits = model(input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.decode(predicted_ids[0])

    del model
    del processor

    if torch.cuda.is_available():
      torch.cuda.empty_cache()

    return transcription


In [8]:
audio_path = "./21144-01.mp3"
audio, sample_rate = librosa.load(audio_path, sr=16000)
print(f"Duración del audio: {len(audio)/sample_rate:.1f} segundos")

Duración del audio: 72.6 segundos


In [9]:
model_name = "facebook/wav2vec2-large-xlsr-53-spanish"
print(f"Ejemplo de transcripción usando: {model_name}")
transcribe_sample_wav2vec(audio, model_name)

Ejemplo de transcripción usando: facebook/wav2vec2-large-xlsr-53-spanish


'fábulaesopo rcppaulinopaul el lobo y el cordero en el templo dándose cuenta de que era perseguido por un lobo un pequeño corderito decidió refugiarse en un templo cercano lo llamó lobo y le dijo que si el sacrificador lo encontraba allí adentro lo enmolaría a su dios mejor así replicó el cordero prefiero ser víctima para un dios a tener que perecer en tus colmillos si sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor fábula'

In [10]:
model_name = "jonatasgrosman/wav2vec2-large-xlsr-53-spanish"
print(f"Ejemplo de transcripción usando: {model_name}")
transcribe_sample_wav2vec(audio, model_name)

Ejemplo de transcripción usando: jonatasgrosman/wav2vec2-large-xlsr-53-spanish


'las fábulas de sopo grabado para libribox o hereg por paulino w w w  paulinoinfofábula número sesenta y uno el lobo y el cordero en el templodándo se cuenta de que era perseguido por un lobo un pequeño corderito decidió refugiarse en un templo cercano lo llamó lobo y le dijo que si el sacrificador lo encontraba allí adentro lo inmolaría a su dios mejor así replicó el cordero prefiero ser víctima para un dios a tener que perecer en tucolmillossi sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor fin de la fábulaesta es una grabación del dominio publico'

In [11]:
model_name = "facebook/wav2vec2-base-10k-voxpopuli-ft-es"
print(f"Ejemplo de transcripción usando: {model_name}")
transcribe_sample_wav2vec(audio, model_name)

Ejemplo de transcripción usando: facebook/wav2vec2-base-10k-voxpopuli-ft-es


'las faulas de sopo gravado para el libri vox punto o erg por paulino doble vé doble vé doble vé punto paulino puto info faula número sesenta y uno el lobo y el cordero en el templo tando se cuenta de que era perseguido por un lobo un pequeño cordedito desidió refugiarse en un templio cercano lo lamó lobo y le dijo que si el sacrificador lo encontraba allía dentro lo enmolaría a su dios mejor así eel cordero crefiero a ser víctima para un dios a tener que pereser en tus colmillos si sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor fin de la fabula esta es una gravación del dominio público'

### Ejemplo SpeechSeq2Seq



#### IBM Granite

In [19]:
model_name = 'ibm-granite/granite-speech-4.1-2b'

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_name, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
).to(device)

print(f"model {model_name} loaded")

Loading weights:   0%|          | 0/954 [00:00<?, ?it/s]

In [20]:
# Apply standard chat template mapping
prompt = "<|user|>\n<|audio|>\nTranscribe the audio.<|assistant|>\n"

# Process both audio and text prompt together
inputs = processor(
  audio=audio,
  text=prompt,
  sampling_rate=16000,
  return_tensors="pt"
).to(device, dtype=torch_dtype)

# Generate output with safety thresholds
with torch.no_grad():
  output_ids = model.generate(
    **inputs,
    max_new_tokens=400, # Stops infinite loop hangs
    do_sample=False,    # Greedy decoding for stable transcription
  )

# Post-process and decode text boundaries
# We strip the prompt length to only return the model's new answer tokens
input_length = inputs.input_ids.shape[1]
generated_text = processor.decode(output_ids[0][input_length:], skip_special_tokens=True)

# Clean up memory allocations
del model
del processor
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [21]:
generated_text

'fábula número sesenta y uno el lobo y el cordero en el templo dándose cuenta de que era perseguido por un lobo un pequeño corderito decidió refugiarse en un templo cercano lo llamó al lobo y le dijo que si el sacrificador lo encontraba allí adentro lo inmolaría a su dios mejor así replicó el cordero prefiero ser víctima para un dios a tener que perecer en tus colmillos si sin remedio vamos a ser sacrificados más nos vale que sea con el mayor honor fábula'

#### Open AI Whisper

In [25]:
model_id="openai/whisper-large-v3-turbo"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)

model.to(device)
print(f"model {model_id} loaded")

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

model openai/whisper-large-v3-turbo loaded


In [26]:
processor = AutoProcessor.from_pretrained(model_id)

In [27]:
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=torch_dtype,
    device=device,

    # Tells the pipeline to automatically cut long audio into 30s chunks
    chunk_length_s=30,

    # Passes the required parameter down to the Whisper generation config
    generate_kwargs={"return_timestamps": True, "language": "spanish"}
)

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [28]:
result = pipe(audio)

In [29]:
result

{'text': ' Las fábulas de Esopo. Grabado para LibriVox.org por Paulino. www.paulino.info Fábula número 61. El lobo y el cordero en el templo. Dándose cuenta de que era perseguido por un lobo, un pequeño corderito decidió refugiarse en un templo cercano. Lo llamó lobo y le dijo que si el sacrificador lo encontraba allí adentro, lo enmolaría a su dios. Mejor así, replicó el cordero, prefiero ser víctima para un dios a tener que perecer en tus colmillos. Si sin remedio vamos a ser sacrificados, más nos vale que sea con el mayor honor. Fin de la fábula Esta es una grabación del dominio público.'}

### Audio a Texto Mejor Modelo

In [31]:
BATCH_SIZE = 16
dir_path = Path('.')

mp3_files = list(dir_path.glob('*.mp3'))

def audio_generator(files):
  for file in files:
    yield str(file)

diccionario_audio_texto = {}

print(f"Procesando por batches {len(mp3_files)} archivos")

# El modelo es capaz de leer el archivo mp3 con solo la ruta
results = pipe(audio_generator(mp3_files), batch_size=BATCH_SIZE)

for file, output in zip(mp3_files, results):
    diccionario_audio_texto[file.name] = output['text']


Procesando por batches 10 archivos


In [32]:
for key, value in diccionario_audio_texto.items():
    print(f"{key}: {value}")

21144-25.mp3:  Las fábulas de Esopo, grabada para LibreVox.org, fábula número 85, El perro y el carnicero. Penetró un perro en una carnicería y notando que el carnicero estaba muy ocupado con sus clientes, cogió un trozo de carne y salió corriendo. Se volvió el carnicero y viéndole huir y sin poder hacer en nada exclamó, Oye amigo, allí donde te encuentre no dejaré de mirarte. No esperes a que suceda un accidente para pensar en cómo evitarlo. Fin de fábula. Esta grabación es de dominio público.
21144-06.mp3:  Las fábulas de Esopo, grabado para LibriVox.org por Alejandro González Calderón. Fábula número 66, El lobo y el asno. Un lobo fue elegido rey entre sus congéneres y decretó una ley ordenando que lo que cada uno capturase en la casa lo pusiera en común y lo repartiese por partes iguales entre todos. De esta manera ya no tendrían los lobos que devorarse unos a otros en épocas de hambre. Pero en eso le escuchó un asno que estaba por ahí cerca y moviendo sus orejas le dijo, magnífica 

# **Ejercicio 2b:**

* #### **Eliminar el inicio y final comunes de los textos extraídos de cada fábula.**

* #### **Sugerencia:** Pueden guardar esta información en un archivo tipo JSON, para que al estar probando diferentes opciones en los ejercicios siguientes, puedan recuperar rápidamente la información de cada video/fábula.

In [33]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
import json

diccionario_path = 'diccionario_audio_texto.json'
if Path(diccionario_path).exists():
  print("leyendo diccionario de fábulas de archivo")
  with open(diccionario_path, 'r') as fp:
      diccionario_audio_texto = json.load(fp)
else:
  print("creando diccionario de fábulas")
  with open(diccionario_path, 'w') as fp:
      json.dump(diccionario_audio_texto, fp)

creando diccionario de fábulas


In [34]:
# limpieza de texto
import re

def clean_text(fabula):
  fabula = re.sub(r"^\D+\d+[^a-z]*", "", fabula, flags=re.IGNORECASE)
  fabula = re.sub(r"fin de.+", "", fabula, flags=re.IGNORECASE)
  return fabula

diccionario_limpio = {}
for key, value in diccionario_audio_texto.items():
  diccionario_limpio[key] = clean_text(value).strip()

In [35]:
diccionario_path = 'diccionario_audio_texto_limpio.json'
if Path(diccionario_path).exists():
  print("leyendo diccionario limpio de fábulas de archivo")
  with open(diccionario_path, 'r') as fp:
      diccionario_limpio = json.load(fp)
else:
  print("creando diccionario limpio de fábulas")
  with open(diccionario_path, 'w') as fp:
      json.dump(diccionario_limpio, fp)

creando diccionario limpio de fábulas


In [60]:
for key, value in diccionario_limpio.items():
    print(f"{key}: {value}")

21144-25.mp3: El perro y el carnicero. Penetró un perro en una carnicería y notando que el carnicero estaba muy ocupado con sus clientes, cogió un trozo de carne y salió corriendo. Se volvió el carnicero y viéndole huir y sin poder hacer en nada exclamó, Oye amigo, allí donde te encuentre no dejaré de mirarte. No esperes a que suceda un accidente para pensar en cómo evitarlo.
21144-06.mp3: El lobo y el asno. Un lobo fue elegido rey entre sus congéneres y decretó una ley ordenando que lo que cada uno capturase en la casa lo pusiera en común y lo repartiese por partes iguales entre todos. De esta manera ya no tendrían los lobos que devorarse unos a otros en épocas de hambre. Pero en eso le escuchó un asno que estaba por ahí cerca y moviendo sus orejas le dijo, magnífica idea ha brotado de tu corazón, pero ¿por qué has escondido todo tu botín en tu cueva,levalo a la comunidad y repártelo también como lo has decretado. El lobo, descubierto y confundido, derogó su ley. Si alguna vez llegas 

# **Ejercicio 3:**

* #### **Apliquen el proceso de limpieza que consideren adecuado.**

* #### **Justifiquen los pasos de limpieza utilizados. Tomen en cuenta que el texto extraído de cada fábula es relativamente pequeño.**

* #### **En caso de que decidan no aplicar esta etapa de limpieza, deberán justificarlo.**

In [38]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

nltk.download('stopwords')
stop_words = set(stopwords.words(['spanish']))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [53]:
fabulas_tokenizadas_dict = {}

for k, v in diccionario_limpio.items():
  words = v.split()
  filtered_words = [
    re.sub(r'[^a-záéíóúñü]', '', w, flags=re.IGNORECASE)
    for word in words if (w := word.lower()) not in stop_words and len(word) >= 3
  ]
  fabulas_tokenizadas_dict[k] = filtered_words

In [56]:
# Monstrando los primeros 5 tokens de cada fabula
fabulas_tokenizadas_dict = dict(sorted(fabulas_tokenizadas_dict.items()))
for key, value in fabulas_tokenizadas_dict.items():
    print(f"{key} (len: {len(value)}): {value[0:5]}")

21144-01.mp3 (len: 40): ['lobo', 'cordero', 'templo', 'dándose', 'cuenta']
21144-04.mp3 (len: 64): ['lobo', 'grulla', 'lobo', 'comía', 'hueso']
21144-05.mp3 (len: 49): ['lobo', 'caballo', 'pasaba', 'lobo', 'sembrado']
21144-06.mp3 (len: 56): ['lobo', 'asno', 'lobo', 'elegido', 'rey']
21144-14.mp3 (len: 33): ['lobo', 'cabrito', 'encerrado', 'protegido', 'seguridad']
21144-22.mp3 (len: 37): ['perro', 'almeja', 'perro', 'acostumbrados', 'comer']
21144-24.mp3 (len: 56): ['perro', 'reflejo', 'río', 'badiaba', 'perro']
21144-25.mp3 (len: 33): ['perro', 'carnicero', 'penetró', 'perro', 'carnicería']
21144-26.mp3 (len: 45): ['perro', 'campanilla', 'perro', 'acostumbraba', 'morder']
21144-27.mp3 (len: 37): ['perro', 'perseguía', 'león', 'perro', 'casa']


# **Ejercicio 4:**

In [64]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.parsing.preprocessing import preprocess_string

In [87]:
documents = [doc for doc in fabulas_tokenizadas_dict.values()]
documents[0][0:5]

['lobo', 'cordero', 'templo', 'dándose', 'cuenta']

In [90]:
diccionario = corpora.Dictionary(documents)
corpus = [diccionario.doc2bow(texto) for texto in documents]
print("Bag of Words representation of first doc (first 5 elements):", corpus[0][0:5])

Bag of Words representation of first doc (first 5 elements): [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1)]


In [117]:
num_topics = 1

# convert previous for loop into a listh comprehension
lda_list = [
  LdaModel(
      corpus=[c],
      id2word=diccionario,
      num_topics=num_topics,
      random_state=42,
      passes=10,
      alpha='symmetric',
      per_word_topics=True
  ) for c in corpus
]

In [118]:
topics_gensim=[]
for i in range(len(lda_list)):
  tg = lda_list[i].print_topics(num_words=30)
  topics_gensim.append(tg)
  print(f"Fábula {i+1}:\n  {tg}")


Fábula 1:
  [(0, '0.010*"lobo" + 0.008*"templo" + 0.008*"cordero" + 0.008*"ser" + 0.008*"dios" + 0.005*"cercano" + 0.005*"así" + 0.005*"adentro" + 0.005*"cuenta" + 0.005*"decidió" + 0.005*"dándose" + 0.005*"encontraba" + 0.005*"honor" + 0.005*"colmillos" + 0.005*"corderito" + 0.005*"allí" + 0.005*"sacrificados" + 0.005*"llamó" + 0.005*"enmolaría" + 0.005*"mayor" + 0.005*"refugiarse" + 0.005*"remedio" + 0.005*"mejor" + 0.005*"perecer" + 0.005*"vale" + 0.005*"sacrificador" + 0.005*"tener" + 0.005*"replicó" + 0.005*"víctima" + 0.005*"vamos"')]
Fábula 2:
  [(0, '0.012*"lobo" + 0.010*"hueso" + 0.010*"paga" + 0.010*"grulla" + 0.007*"boca" + 0.007*"cabeza" + 0.007*"pidió" + 0.007*"garganta" + 0.005*"corría" + 0.005*"corruptos" + 0.005*"convenida" + 0.005*"correr" + 0.005*"crees" + 0.005*"busca" + 0.005*"atragantó" + 0.005*"aceptó" + 0.005*"malvados" + 0.005*"mucha" + 0.005*"hagas" + 0.005*"enseguida" + 0.005*"favores" + 0.005*"entonces" + 0.005*"haber" + 0.005*"encontró" + 0.005*"no" + 0.005*

In [121]:
def get_clean_topics(model, num_words=30):
    raw_words = []
    # Extract raw topic data from the trained model
    raw_topics = model.show_topics(num_topics=-1, num_words=num_words, formatted=False)

    for topic_id, word_list in raw_topics:
        # Each word_list contains tuples of (word, weight)
        raw_words = [word for word, weight in word_list]

    return raw_words

In [132]:
words_lda = []
for i in range(len(lda_list)):
  words_lda.append(get_clean_topics(lda_list[i]))

words_lda[0][0:5]

['lobo', 'templo', 'cordero', 'ser', 'dios']

In [136]:
words_lda_str = ''
for i in range(len(words_lda)):
  words_lda_str += ' '.join(words_lda[i]) + '\n'

print(words_lda_str)

lobo templo cordero ser dios cercano así adentro cuenta decidió dándose encontraba honor colmillos corderito allí sacrificados llamó enmolaría mayor refugiarse remedio mejor perecer vale sacrificador tener replicó víctima vamos
lobo hueso paga grulla boca cabeza pidió garganta corría corruptos convenida correr crees busca atragantó aceptó malvados mucha hagas enseguida favores entonces haber encontró no nunca introdujo sacado partes auxilio
caballo cebada lobo complacer dejó encontró comérsela debe lobos llevó hallado ruido preferido dejado gusto gran sino agradaba vez cantidad camino pasaba comida mejor bueno comentándole campo oídos parezca malvado
lobo ley asno cada capturase poder todos primero escuchó cumplir ahí moviendo leyes casa corazón común legislar lobos partes alguna llegas decretado comunidad cerca hambre descubierto rey repártelo brotado épocas
lobo cabrito sino sitio poderosos valor ampliamente vio encerrado encuentras enfrentamiento infeliz arrogante burlándose comenzó

In [139]:
# save words_lda_str if not exist
if not Path('words_lda_str.txt').exists():
  print("Creando archivo de palabras resultantes")
  with open('words_lda_str.txt', 'w') as f:
    f.write(words_lda_str)
else:
  print("Cargando archivo de palabras resultantes")
  words_lda_str = ''
  with open('words_lda_str.txt', 'r') as f:
    words_lda_str = f.read()

Cargando archivo de palabras resultantes


In [141]:
print(words_lda_str)

lobo templo cordero ser dios cercano así adentro cuenta decidió dándose encontraba honor colmillos corderito allí sacrificados llamó enmolaría mayor refugiarse remedio mejor perecer vale sacrificador tener replicó víctima vamos
lobo hueso paga grulla boca cabeza pidió garganta corría corruptos convenida correr crees busca atragantó aceptó malvados mucha hagas enseguida favores entonces haber encontró no nunca introdujo sacado partes auxilio
caballo cebada lobo complacer dejó encontró comérsela debe lobos llevó hallado ruido preferido dejado gusto gran sino agradaba vez cantidad camino pasaba comida mejor bueno comentándole campo oídos parezca malvado
lobo ley asno cada capturase poder todos primero escuchó cumplir ahí moviendo leyes casa corazón común legislar lobos partes alguna llegas decretado comunidad cerca hambre descubierto rey repártelo brotado épocas
lobo cabrito sino sitio poderosos valor ampliamente vio encerrado encuentras enfrentamiento infeliz arrogante burlándose comenzó

* #### **5a: Mediante el LLM que hayan seleccionado, generar un único enunciado que describa o resuma cada fábula.**

* #### **5b: Mediante el LLM que hayan seleccionado, generar tres posibles enunciados diferentes relacionados con la historia de la fábula.**

* #### **Sugerencia:** En realidad los dos incisos a y b se pueden obtener con un solo prompt que solicite la información y el formato correspondiente para cada una de estas partes. Por ejemplo, para cada fábula la salida puede ser un primer enunciado genérico que resume o describe dicha temática; seguido de tres enunciados, cada uno hablando sobre una situación o parte diferente de la fábula.

In [146]:
# Save
import os
from google import genai

gemini_token = userdata.get('GEMINI_API_KEY')

client = genai.Client(
    api_key=gemini_token
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Resume la moraleja de la fábula del lobo y el cordero."
)

print(response.text)

La moraleja principal de la fábula del lobo y el cordero es que **quien tiene el poder y la intención de hacer daño, siempre encontrará un pretexto, por más ilógico o injusto que sea, para justificar sus acciones y oprimir al más débil.**

En esencia, demuestra que:
*   **La razón y la lógica son inútiles frente a la fuerza bruta y la malicia premeditada.** El agresor no busca la verdad, sino una excusa para ejercer su voluntad.
*   **La tiranía no necesita argumentos válidos; solo necesita un pretexto.**


In [ ]:
# Incluyan a continuación todas las celdas (de código o texto) que deseen...

prompt_template = """
Eres un asistente especializado en análisis de textos narrativos y fábulas.

Recibirás una línea de texto que representa una fábula procesada previamente con limpieza de texto y LDA.
La línea contiene palabras clave, no necesariamente en orden narrativo.

Tu tarea es inferir, con cautela, el contenido general de la fábula.

Devuelve únicamente un objeto JSON válido con esta estructura:

{{
  "resumen": "Resumen en español de máximo 20 palabras.",
  "subtemas": [
    "Subtema 1",
    "Subtema 2",
    "Subtema 3"
  ]
}}

Reglas:
- El resumen debe tener máximo 20 palabras.
- Los subtemas deben ser diferentes entre sí.
- Los subtemas deben relacionarse con la moraleja, conflicto o enseñanza de la fábula.
- No inventes nombres, eventos ni detalles específicos si no están respaldados por las palabras clave.
- Si reconoces una fábula clásica, puedes usar ese conocimiento para mejorar la interpretación.
- No incluyas explicaciones adicionales.
- No uses formato Markdown.
- Devuelve solo JSON válido.

Texto de la fábula:
"{linea}"
"""


In [153]:
# Construir texto númerado

fabulas_texto = "\n\n".join(
    f"[{i+1}]\n{linea}"
    for i, linea in enumerate(words_lda_str.splitlines())
)

fabulas_texto


'[1]\nlobo templo cordero ser dios cercano así adentro cuenta decidió dándose encontraba honor colmillos corderito allí sacrificados llamó enmolaría mayor refugiarse remedio mejor perecer vale sacrificador tener replicó víctima vamos\n\n[2]\nlobo hueso paga grulla boca cabeza pidió garganta corría corruptos convenida correr crees busca atragantó aceptó malvados mucha hagas enseguida favores entonces haber encontró no nunca introdujo sacado partes auxilio\n\n[3]\ncaballo cebada lobo complacer dejó encontró comérsela debe lobos llevó hallado ruido preferido dejado gusto gran sino agradaba vez cantidad camino pasaba comida mejor bueno comentándole campo oídos parezca malvado\n\n[4]\nlobo ley asno cada capturase poder todos primero escuchó cumplir ahí moviendo leyes casa corazón común legislar lobos partes alguna llegas decretado comunidad cerca hambre descubierto rey repártelo brotado épocas\n\n[5]\nlobo cabrito sino sitio poderosos valor ampliamente vio encerrado encuentras enfrentamient

In [154]:
prompt = f"""
Eres un asistente especializado en análisis de fábulas.

Contexto:
- Cada línea representa una fábula diferente.
- Las palabras fueron obtenidas después de limpieza de texto y modelado LDA.
- El orden de las palabras no necesariamente representa el orden original de la historia.
- Debes inferir el contenido general de la fábula a partir de las palabras clave.
- Si reconoces una fábula clásica puedes usar ese conocimiento para mejorar la interpretación.

Para cada fábula genera:
1. Un resumen de máximo 20 palabras.
2. Tres subtemas diferentes.

Devuelve ÚNICAMENTE un arreglo JSON válido.

Formato esperado:

[
  {{
    "id": 1,
    "resumen": "...",
    "subtemas": [
      "...",
      "...",
      "..."
    ]
  }}
]

Fábulas:

{fabulas_texto}
"""

In [155]:
print(prompt)


Eres un asistente especializado en análisis de fábulas.

Contexto:
- Cada línea representa una fábula diferente.
- Las palabras fueron obtenidas después de limpieza de texto y modelado LDA.
- El orden de las palabras no necesariamente representa el orden original de la historia.
- Debes inferir el contenido general de la fábula a partir de las palabras clave.
- Si reconoces una fábula clásica puedes usar ese conocimiento para mejorar la interpretación.

Para cada fábula genera:
1. Un resumen de máximo 20 palabras.
2. Tres subtemas diferentes.

Devuelve ÚNICAMENTE un arreglo JSON válido.

Formato esperado:

[
  {
    "id": 1,
    "resumen": "...",
    "subtemas": [
      "...",
      "...",
      "..."
    ]
  }
]

Fábulas:

[1]
lobo templo cordero ser dios cercano así adentro cuenta decidió dándose encontraba honor colmillos corderito allí sacrificados llamó enmolaría mayor refugiarse remedio mejor perecer vale sacrificador tener replicó víctima vamos

[2]
lobo hueso paga grulla boca 

In [156]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

```json
[
  {
    "id": 1,
    "resumen": "Un lobo intenta sacar a un cordero refugiado en un templo para devorarlo, pero el cordero lo refuta astutamente.",
    "subtemas": [
      "La astucia frente al engaño",
      "La protección de lo sagrado",
      "La manipulación de la fe"
    ]
  },
  {
    "id": 2,
    "resumen": "Un lobo pide ayuda a una grulla para quitar un hueso de su garganta, prometiendo pago que luego niega.",
    "subtemas": [
      "La ingratitud",
      "El peligro de ayudar a los malvados",
      "La falsa promesa"
    ]
  },
  {
    "id": 3,
    "resumen": "Un lobo intenta engañar a un caballo para que coma algo que no le agrada, revelando sus verdaderas intenciones.",
    "subtemas": [
      "La hipocresía",
      "La falsa amabilidad",
      "La percepción de la maldad"
    ]
  },
  {
    "id": 4,
    "resumen": "Un lobo, como parte de una comunidad, legisla una ley que favorece a los poderosos, especialmente en tiempos de hambre.",
    "subtemas": [
      "La 

In [159]:
# Parseamos la respuesta de gemini
text = response.text.strip()

if text.startswith("```json"):
    text = text[7:]

if text.endswith("```"):
    text = text[:-3]

text = text.strip()

data = json.loads(text)

In [161]:
# if archivo de respuesta de ai no existe crearlo con data
if not Path('respuesta_ai.json').exists():
  print("Creando archivo de respuesta de AI")
  with open('respuesta_ai.json', 'w') as f:
    json.dump(data, f)
else:
  print("Cargando archivo de respuesta de AI")
  with open('respuesta_ai.json', 'r') as f:
    data = json.load(f)

Creando archivo de respuesta de AI


# **Ejercicio 6:**

* #### **Incluyan sus conclusiones de la actividad audio-a-texto:**



None

# **Fin de la actividad LDA y LMM: audio-a-texto**